# 🧠 NLP Assignment – 4: Fine-Tuning BERT on IMDB Movie Reviews

**Objective:** Fine-tune a pre-trained BERT model on the IMDB Movie Reviews dataset for binary sentiment classification (Positive / Negative).

---

**Pipeline Overview:**
```
Raw Data → Preprocessing → Tokenization → Model Training → Evaluation → Experiment Comparison
```

| Item | Detail |
|---|---|
| **Dataset** | IMDB Movie Reviews (50,000 reviews) |
| **Task** | Binary Sentiment Classification |
| **Base Model** | `bert-base-uncased` |
| **Framework** | PyTorch + Hugging Face Transformers |
| **Experiments** | Frozen BERT vs Last-2-Layers vs Full Fine-Tune |

## Step 1: Install Required Libraries

In [ ]:
# Install all required packages
# transformers  – Hugging Face model hub, tokenizers, and training utilities
# datasets      – Hugging Face datasets library (includes IMDB)
# scikit-learn  – Evaluation metrics (accuracy, precision, recall, F1, confusion matrix)
# seaborn       – Heatmap visualization for confusion matrix
!pip install transformers datasets scikit-learn seaborn --quiet

print("✅ All libraries installed!")

## Step 2: Import Libraries

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings("ignore")

# Hugging Face — model, tokenizer, dataset
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup   # BONUS: learning rate scheduler
)
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    classification_report
)
from sklearn.model_selection import train_test_split

# Use GPU if available (CUDA), else fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Libraries imported!")
print(f"🖥️  Device: {device}")

## Step 3: Load Dataset

We use the **IMDB Movie Reviews** dataset via Hugging Face `datasets`.
- 25,000 training reviews + 25,000 test reviews
- Labels: `0` = Negative, `1` = Positive

We take a **subset of 3,000 samples** for faster training in this notebook environment. Increase `SAMPLE_SIZE` for better performance.

In [ ]:
# Load IMDB dataset from Hugging Face (no Kaggle account needed)
print("⏳ Loading IMDB dataset...")
raw_dataset = load_dataset("imdb")

# Convert to pandas DataFrames for easy manipulation
train_df = pd.DataFrame(raw_dataset["train"])
test_df  = pd.DataFrame(raw_dataset["test"])

print(f"✅ Dataset loaded!")
print(f"   Training samples : {len(train_df):,}")
print(f"   Test samples     : {len(test_df):,}")
print(f"\n📋 Sample rows:")
train_df.head(3)

## Step 4: Data Preprocessing

Text cleaning steps:
1. Remove HTML tags (IMDB reviews contain `<br />` tags)
2. Remove URLs
3. Remove special characters and punctuation
4. Normalize whitespace
5. Convert to lowercase

In [ ]:
def clean_text(text):
    """
    Cleans raw review text for BERT tokenization.

    Steps:
    - Strip HTML tags (e.g., <br />) common in IMDB reviews
    - Remove URLs
    - Remove non-alphanumeric characters (keep spaces)
    - Normalize multiple spaces to single space
    - Strip leading/trailing whitespace
    - Lowercase the text
    """
    text = re.sub(r"<.*?>", " ", text)           # Remove HTML tags
    text = re.sub(r"http\S+|www\S+", " ", text)  # Remove URLs
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)  # Keep only alphanumeric + spaces
    text = re.sub(r"\s+", " ", text).strip()      # Normalize whitespace
    text = text.lower()                            # Lowercase
    return text

# Apply cleaning to both train and test sets
print("🧹 Cleaning text data...")
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"]  = test_df["text"].apply(clean_text)

# Check for missing values after cleaning
print(f"   Missing values in train: {train_df['clean_text'].isnull().sum()}")
print(f"   Missing values in test : {test_df['clean_text'].isnull().sum()}")

# Before / After comparison
print("\n📝 Before cleaning:")
print(train_df["text"].iloc[0][:200])
print("\n📝 After cleaning:")
print(train_df["clean_text"].iloc[0][:200])

## Step 5: Data Splitting

We use a **subset** for efficiency and split into:
- **Train**: 70%
- **Validation**: 15%
- **Test**: 15%

Using `stratify=y` ensures balanced class distribution in every split.

In [ ]:
# Subset size — increase for better accuracy (requires more compute time)
SAMPLE_SIZE = 3000

# Sample equally from both classes (1500 positive + 1500 negative)
sampled_df = train_df.groupby("label").sample(n=SAMPLE_SIZE // 2, random_state=42).reset_index(drop=True)

X = sampled_df["clean_text"].tolist()   # Text inputs
y = sampled_df["label"].tolist()         # Labels: 0=Negative, 1=Positive

# Split into Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Split Temp into Validation (50%) and Test (50%) → each is 15% of total
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"✅ Data split complete:")
print(f"   Train      : {len(X_train):,} samples")
print(f"   Validation : {len(X_val):,} samples")
print(f"   Test       : {len(X_test):,} samples")
print(f"\n📊 Label distribution in train:")
print(pd.Series(y_train).value_counts().rename({0: 'Negative', 1: 'Positive'}))

## Step 6: Tokenization

BERT requires specific input format:
- **`input_ids`** — Token IDs from BERT's vocabulary
- **`attention_mask`** — 1 for real tokens, 0 for padding
- **`token_type_ids`** — Segment IDs (0 for single-sentence tasks)

We set `max_length=128` (BERT supports up to 512; 128 is sufficient for most reviews and trains faster).

In [ ]:
# Load the bert-base-uncased tokenizer
# 'uncased' means all text is lowercased — matches our preprocessing
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 128   # Max token sequence length per review
BATCH_SIZE = 16    # Number of samples processed per gradient update

print(f"⏳ Loading tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokenizer loaded!")


def tokenize_texts(texts, tokenizer, max_length):
    """
    Tokenizes a list of texts using the BERT tokenizer.

    Returns:
        input_ids      – Token ID sequences
        attention_masks – Mask indicating real vs padded tokens
    """
    encoded = tokenizer(
        texts,
        max_length=max_length,       # Truncate/pad to this length
        padding="max_length",        # Pad shorter sequences
        truncation=True,             # Truncate longer sequences
        return_tensors="pt"          # Return as PyTorch tensors
    )
    return encoded["input_ids"], encoded["attention_mask"]


print("⏳ Tokenizing splits...")

# Tokenize all three splits
train_ids, train_masks = tokenize_texts(X_train, tokenizer, MAX_LEN)
val_ids,   val_masks   = tokenize_texts(X_val,   tokenizer, MAX_LEN)
test_ids,  test_masks  = tokenize_texts(X_test,  tokenizer, MAX_LEN)

# Convert labels to tensors
train_labels = torch.tensor(y_train)
val_labels   = torch.tensor(y_val)
test_labels  = torch.tensor(y_test)

print(f"✅ Tokenization complete!")
print(f"   input_ids shape (train): {train_ids.shape}")
print(f"   Sample token IDs       : {train_ids[0][:15].tolist()} ...")

## Step 7: Create DataLoaders

`TensorDataset` bundles input IDs, attention masks, and labels together.
`DataLoader` handles batching, shuffling (train only), and iteration.

In [ ]:
# Bundle tensors into TensorDatasets
train_dataset = TensorDataset(train_ids, train_masks, train_labels)
val_dataset   = TensorDataset(val_ids,   val_masks,   val_labels)
test_dataset  = TensorDataset(test_ids,  test_masks,  test_labels)

# Create DataLoaders
# shuffle=True for training ensures different batch ordering each epoch
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ DataLoaders created!")
print(f"   Train batches      : {len(train_loader)}")
print(f"   Validation batches : {len(val_loader)}")
print(f"   Test batches       : {len(test_loader)}")

## Step 8: Define Helper Functions — Train, Evaluate, and Plot

We define reusable functions so all three experiments use identical code with different model configurations.

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    """
    Runs one full training epoch.
    Returns average loss and accuracy over all training batches.
    """
    model.train()   # Enable dropout and batch norm for training
    total_loss, correct, total = 0, 0, 0

    for batch in loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        optimizer.zero_grad()  # Clear gradients from previous step

        # Forward pass: compute logits and loss simultaneously
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()               # Backpropagation — compute gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping prevents exploding gradients
        optimizer.step()              # Update model weights
        scheduler.step()              # BONUS: update learning rate

        # Track metrics
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, device):
    """
    Evaluates model on a DataLoader.
    Returns: loss, accuracy, all true labels, all predicted labels.
    """
    model.eval()   # Disable dropout for deterministic evaluation
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():  # No gradient computation needed for evaluation
        for batch in loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_labels, all_preds


def compute_metrics(true_labels, pred_labels, experiment_name):
    """
    Computes and prints all evaluation metrics.
    Returns a dict of metric values.
    """
    acc  = accuracy_score(true_labels, pred_labels)
    prec = precision_score(true_labels, pred_labels)
    rec  = recall_score(true_labels, pred_labels)
    f1   = f1_score(true_labels, pred_labels)

    print(f"\n{'='*50}")
    print(f"  📊 Results: {experiment_name}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"{'='*50}")
    print(classification_report(true_labels, pred_labels, target_names=["Negative", "Positive"]))

    return {"Experiment": experiment_name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1 Score": f1}


def plot_confusion_matrix(true_labels, pred_labels, title):
    """
    Plots a heatmap confusion matrix.
    """
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Negative", "Positive"],
        yticklabels=["Negative", "Positive"]
    )
    plt.title(f"Confusion Matrix — {title}", fontsize=13, fontweight="bold")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()


def plot_training_curves(train_losses, val_losses, train_accs, val_accs, title):
    """
    Plots loss and accuracy curves over epochs.
    """
    epochs = range(1, len(train_losses) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss curve
    axes[0].plot(epochs, train_losses, "b-o", label="Train Loss")
    axes[0].plot(epochs, val_losses,   "r-o", label="Val Loss")
    axes[0].set_title(f"Loss — {title}"); axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss"); axes[0].legend()

    # Accuracy curve
    axes[1].plot(epochs, train_accs, "b-o", label="Train Acc")
    axes[1].plot(epochs, val_accs,   "r-o", label="Val Acc")
    axes[1].set_title(f"Accuracy — {title}"); axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy"); axes[1].legend()

    plt.tight_layout()
    plt.show()


print("✅ Helper functions defined!")

## Step 9: Define Experiment Runner

A single function that runs any experiment given a model freeze configuration.
This avoids code duplication across the three experiments.

In [ ]:
EPOCHS    = 3     # Number of training epochs
LR        = 2e-5  # Learning rate as specified in assignment
NUM_LABELS = 2    # Binary classification: Negative (0) / Positive (1)


def run_experiment(experiment_name, freeze_mode="none"):
    """
    Loads a fresh BERT model, applies freeze_mode, trains, and evaluates.

    Args:
        experiment_name (str): Human-readable name for logging.
        freeze_mode (str): One of:
            'all'    – Freeze all BERT layers (train only classifier head)
            'last2'  – Freeze all layers EXCEPT last 2 transformer blocks
            'none'   – Full fine-tuning (all layers trainable)

    Returns:
        dict of evaluation metrics on the test set.
    """
    print(f"\n{'#'*60}")
    print(f"  🔬 EXPERIMENT: {experiment_name}")
    print(f"  Freeze Mode: {freeze_mode}")
    print(f"{'#'*60}")

    # --- 9a: Load a fresh pre-trained BERT model ---
    # num_labels=2 adds a linear classification head on top of [CLS] token representation
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS
    )
    model.to(device)

    # --- 9b: Apply layer freezing based on experiment ---
    if freeze_mode == "all":
        # Freeze all BERT encoder layers — only the classification head trains
        for name, param in model.bert.named_parameters():
            param.requires_grad = False
        print("  ❄️  All BERT layers frozen. Only classifier head is trainable.")

    elif freeze_mode == "last2":
        # Freeze embeddings + first 10 transformer layers (BERT-base has 12 total)
        # Unfreeze layers 10, 11 (0-indexed) and the pooler + classifier
        for name, param in model.bert.named_parameters():
            # Keep layers 10 and 11 trainable; freeze everything else
            if not any(f"encoder.layer.{i}" in name for i in [10, 11]):
                param.requires_grad = False
        print("  🔓 Last 2 BERT encoder layers + classifier unfrozen.")

    else:  # 'none' — full fine-tuning
        print("  🔓 All layers trainable (full fine-tuning).")

    # Count trainable parameters to verify freeze worked
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  📦 Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    # --- 9c: Set up AdamW optimizer ---
    # AdamW adds weight decay (L2 regularization) separately from gradient updates
    # We only pass parameters that require gradients
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=0.01   # L2 regularization to prevent overfitting
    )

    # --- BONUS: Linear warmup + decay scheduler ---
    # Gradually increases LR during warmup, then linearly decays to 0
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(0.1 * total_steps)  # 10% of steps for warmup
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # --- 9d: Training loop ---
    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []
    best_val_loss = float("inf")
    patience_counter = 0
    PATIENCE = 2  # BONUS: Early stopping patience

    for epoch in range(1, EPOCHS + 1):
        print(f"\n  Epoch {epoch}/{EPOCHS}")

        t_loss, t_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
        v_loss, v_acc, _, _ = evaluate(model, val_loader, device)

        train_losses.append(t_loss); val_losses.append(v_loss)
        train_accs.append(t_acc);   val_accs.append(v_acc)

        print(f"  Train — Loss: {t_loss:.4f} | Acc: {t_acc:.4f}")
        print(f"  Val   — Loss: {v_loss:.4f} | Acc: {v_acc:.4f}")

        # BONUS: Early stopping — stop if val loss doesn't improve
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  ⏹️  Early stopping triggered at epoch {epoch}.")
                break

    # --- 9e: Plot training curves ---
    plot_training_curves(train_losses, val_losses, train_accs, val_accs, experiment_name)

    # --- 9f: Evaluate on test set ---
    _, _, true_labels, pred_labels = evaluate(model, test_loader, device)

    # --- 9g: Compute and display metrics ---
    metrics = compute_metrics(true_labels, pred_labels, experiment_name)

    # --- 9h: Plot confusion matrix ---
    plot_confusion_matrix(true_labels, pred_labels, experiment_name)

    return metrics


print("✅ Experiment runner defined!")

---
## ⚗️ Experiment 1: Frozen BERT (Classifier Head Only)

**Setup:** All BERT encoder layers are frozen. Only the final linear classification head is trained.

**Expected behavior:** Fast training, lower accuracy — the BERT representations are not adapted to this task.

In [ ]:
results_exp1 = run_experiment(
    experiment_name="Exp 1: Frozen BERT (Classifier Only)",
    freeze_mode="all"
)

---
## ⚗️ Experiment 2: Fine-Tune Last 2 Layers of BERT

**Setup:** BERT embedding layer and first 10 transformer blocks are frozen. Only the last 2 encoder blocks (layers 10 & 11) + classifier head are trainable.

**Expected behavior:** Better accuracy than Exp 1 since upper layers adapt to sentiment. Much faster than full fine-tuning.

In [ ]:
results_exp2 = run_experiment(
    experiment_name="Exp 2: Fine-Tune Last 2 Layers",
    freeze_mode="last2"
)

---
## ⚗️ Experiment 3: Full Fine-Tuning (All Layers)

**Setup:** All BERT layers are trainable — the standard fine-tuning approach.

**Expected behavior:** Best accuracy. Slower training but maximum model adaptation.

In [ ]:
results_exp3 = run_experiment(
    experiment_name="Exp 3: Full Fine-Tuning (All Layers)",
    freeze_mode="none"
)

---
## Step 10: Comparison Table — All Experiments

Side-by-side comparison of all three experiments across every metric.

In [ ]:
# Combine all experiment results into a comparison DataFrame
comparison_df = pd.DataFrame([results_exp1, results_exp2, results_exp3])
comparison_df = comparison_df.set_index("Experiment")
comparison_df = comparison_df.round(4)

print("\n📊 Experiment Comparison Table:")
print(comparison_df.to_string())

In [ ]:
# Visualize the comparison as a grouped bar chart
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1 Score"]
comparison_df[metrics_to_plot].plot(
    kind="bar",
    figsize=(12, 6),
    colormap="Set2",
    edgecolor="black"
)
plt.title("Experiment Comparison — BERT Fine-Tuning on IMDB", fontsize=14, fontweight="bold")
plt.xlabel("Experiment")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.ylim(0.5, 1.0)   # Focus y-axis on relevant range
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

---
## Step 11: Analysis and Insights

### Summary of Findings

| Experiment | Trainable Layers | Expected Trade-off |
|---|---|---|
| **Exp 1: Frozen BERT** | Classifier head only | Fastest training, lowest accuracy |
| **Exp 2: Last 2 Layers** | BERT layers 10–11 + classifier | Good balance of speed and accuracy |
| **Exp 3: Full Fine-Tuning** | All 12 layers + classifier | Best accuracy, slowest training |

### Key Observations

1. **Frozen BERT (Exp 1)** performs as a feature extractor — BERT's pre-trained representations are used as-is and only the decision boundary is learned. This is useful when compute is limited but sacrifices task-specific adaptation.

2. **Last-2-Layers Fine-Tuning (Exp 2)** offers a strong middle ground. Upper BERT layers capture more task-specific features and benefit from fine-tuning, while lower layers (which learn general language syntax) remain frozen.

3. **Full Fine-Tuning (Exp 3)** achieves the best performance because every layer adapts to the nuances of sentiment language. However, it uses the most memory and takes the longest to train.

4. **Gradient clipping** (clip_grad_norm_ at 1.0) was essential for stable training — without it, large gradients in early fine-tuning epochs can destabilize the model.

5. **BONUS — LR Scheduler**: Linear warmup + decay helped the model converge more smoothly, especially in full fine-tuning where abrupt large updates to pre-trained weights can be destructive.

### Recommendation

> For real-world deployment with limited resources, **Experiment 2 (Last 2 Layers)** gives the best accuracy-to-cost ratio. For highest possible accuracy with no resource constraints, **Experiment 3 (Full Fine-Tuning)** is preferred.

---
*Assignment NLP – 4 | Data Science Internship | February 2026*